<a href="https://colab.research.google.com/github/tetsufuku81/silicon-light-code/blob/main/noise_shaping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Noise shaping & decimation — visualize quantization noise in the frequency domain.
Companion code for the blog article (slug: noise-shaping-python).

- Pure numpy for the DSP core; matplotlib for plots; seaborn only for cosmetics.
- Deterministic: a fixed RNG seed gives identical figures/numbers on any OS / Colab.
- Plot labels are English on purpose (portable across environments; reusable on Tindie/Medium/LinkedIn).

Run:
    pip install numpy matplotlib seaborn
    python noise_shaping.py
Writes PNGs into ./figures/ and prints the SNR/ENOB table.
"""

import os
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(context="notebook", style="whitegrid")  # cosmetic only; safe to drop
except ImportError:
    pass

# Headless (e.g. a bare WSL shell with no display)? Uncomment to avoid opening windows:
#   import matplotlib; matplotlib.use("Agg")
# In Colab / Jupyter the inline backend shows the figures automatically.

os.makedirs("figures", exist_ok=True)

# ---- reproducibility ----
SEED = 12345
rng = np.random.default_rng(SEED)

# ---- parameters ----
N      = 1 << 16          # samples (power of two -> clean FFT)
NBITS  = 4               # quantizer resolution (deliberately low)
OSR    = 64              # oversampling ratio
FS     = 1.0             # normalized sample rate
CYCLES = 101             # integer, prime-ish -> coherent tone, little FFT leakage
F0     = CYCLES / N * FS # input tone, inside the band of interest
AMP    = 0.5             # amplitude (fraction of full scale) -> headroom for the modulator
BAND   = FS / (2 * OSR)  # band of interest: DC .. fs/(2*OSR)

n = np.arange(N)
x = AMP * np.sin(2 * np.pi * F0 * n)        # full scale is +-1

# ---- uniform quantizer (step DELTA) + TPDF dither ----
DELTA = 2.0 / (2 ** NBITS)

def quantize(u):
    return np.clip(DELTA * np.round(u / DELTA), -1.0, 1.0 - DELTA)

# Triangular (TPDF) dither decorrelates the quantization error -> clean, textbook spectra
dither = (rng.random(N) - rng.random(N)) * DELTA

# (a) direct quantization, no shaping
y_direct = quantize(x + dither)

# (b) 1st-order sigma-delta (error feedback): Y = X + (1 - z^-1) E, so NTF(z) = 1 - z^-1
y_sd = np.empty(N)
e = 0.0
for i in range(N):
    w = x[i] - e                 # subtract the previous error (NO dither on the input path)
    v = quantize(w + dither[i])  # dither the quantizer -> the dither is noise-shaped too
    e = v - w                    # this step's error (dither + quantization error)
    y_sd[i] = v

# ---- windowed-sinc low-pass + decimate by OSR (numpy only) ----
def lowpass_fir(cutoff, numtaps):
    m = np.arange(numtaps) - (numtaps - 1) / 2.0
    h = 2 * cutoff * np.sinc(2 * cutoff * m) * np.hamming(numtaps)
    return h / h.sum()           # unity DC gain

h = lowpass_fir(BAND, 255)

def decimate(sig):
    return np.convolve(sig, h, mode="same")[::OSR]

y_dec = decimate(y_sd)           # decimated shaped output (fs -> fs/OSR)

# ---- spectra (one-sided PSD, dB relative to a full-scale sine) ----
def psd_db(sig):
    w = np.hanning(len(sig))
    w = w / np.sqrt(np.mean(w ** 2))        # keep power
    X = np.fft.rfft(sig * w) / len(sig)
    p = 2.0 * np.abs(X) ** 2                 # one-sided
    f = np.fft.rfftfreq(len(sig), d=1.0)
    ref = AMP ** 2 / 2.0 / (AMP ** 2)        # 0 dB = full-scale sine
    return f, 10 * np.log10(p / 0.5 + 1e-20) # 0.5 = power of a full-scale sine

# ---- in-band SNR / ENOB ----
def snr_enob(sig, f0, band):
    w = np.hanning(len(sig)); w /= np.sqrt(np.mean(w ** 2))
    P = np.abs(np.fft.rfft(sig * w)) ** 2
    f = np.fft.rfftfreq(len(sig), d=1.0)
    k0 = int(np.argmin(np.abs(f - f0)))
    sig_bins = slice(max(k0 - 3, 1), k0 + 4)     # tone + leakage
    inband = f <= band
    inband[0] = False                            # drop DC
    noise = inband.copy(); noise[sig_bins] = False
    snr = 10 * np.log10(P[sig_bins].sum() / P[noise].sum())
    return snr, (snr - 1.76) / 6.02

snr_direct, enob_direct = snr_enob(y_direct, F0, BAND)
snr_sd,     enob_sd     = snr_enob(y_sd,     F0, BAND)
snr_dec,    enob_dec    = snr_enob(y_dec,    F0 * OSR, 0.5)   # decimated: tone at F0*OSR, full band

print(f"{'stage':<28}{'in-band SNR [dB]':>18}{'ENOB [bit]':>14}")
print(f"{'direct quantization':<28}{snr_direct:>18.1f}{enob_direct:>14.1f}")
print(f"{'sigma-delta (shaped)':<28}{snr_sd:>18.1f}{enob_sd:>14.1f}")
print(f"{'after decimation':<28}{snr_dec:>18.1f}{enob_dec:>14.1f}")

# ---- Figure 1: time domain (sine vs 4-bit quantized) ----
seg = slice(0, 400)
plt.figure(figsize=(8, 3.2))
plt.plot(n[seg], x[seg], label="input sine", lw=1.5)
plt.step(n[seg], y_direct[seg], where="mid", label=f"{NBITS}-bit quantized", lw=1.0)
plt.xlabel("sample"); plt.ylabel("amplitude"); plt.legend(loc="upper right")
plt.title("Time domain: quantization staircase"); plt.tight_layout()
plt.savefig("figures/fig1_timedomain.png", dpi=130); plt.show()

# ---- Figure 2: spectra, direct vs shaped (+ NTF theory overlay) ----
f, Pdir = psd_db(y_direct)
_, Psd  = psd_db(y_sd)
# 1st-order NTF magnitude: |1 - e^{-j w}|^2 = 4 sin^2(pi f / fs), scaled to sit on the shaped floor
w_ntf = 2 * np.pi * f / FS
ntf_db = 10 * np.log10(4 * np.sin(w_ntf / 2) ** 2 + 1e-20)
plt.figure(figsize=(8, 4))
plt.semilogx(f[1:], Pdir[1:], label="direct quantization", lw=1.0, alpha=0.8)
plt.semilogx(f[1:], Psd[1:],  label="sigma-delta (shaped)", lw=1.0, alpha=0.8)
plt.semilogx(f[1:], ntf_db[1:] - 55, "--", color="k",
             label=r"$|NTF|^2 = (2\sin(\pi f/f_s))^2$ (theory)", lw=1.2)
plt.axvline(BAND, color="r", ls=":", lw=1.0, label="band edge fs/(2*OSR)")
plt.xlabel("frequency [x fs]"); plt.ylabel("PSD [dB rel. full-scale sine]")
plt.ylim(-140, 0); plt.legend(loc="lower right", fontsize=8)
plt.title("Noise moves out of band — and the theory curve predicts the shape")
plt.tight_layout(); plt.savefig("figures/fig2_spectra.png", dpi=130); plt.show()

# ---- Figure 3: decimated spectrum ----
fd, Pdec = psd_db(y_dec)
plt.figure(figsize=(8, 3.6))
plt.plot(fd, Pdec, lw=1.0)
plt.xlabel("frequency [x fs_decimated]"); plt.ylabel("PSD [dB rel. full-scale sine]")
plt.ylim(-140, 0)
plt.title(f"After decimation: low in-band floor (ENOB ~ {enob_dec:.1f} bit from {NBITS}-bit)")
plt.tight_layout(); plt.savefig("figures/fig3_decimated.png", dpi=130); plt.show()

# ---- Figure 4: in-band SNR vs OSR (measured vs 9 dB/octave) ----
osr_list = np.array([8, 16, 32, 64, 128, 256])
snr_meas = [snr_enob(y_sd, F0, FS / (2 * o))[0] for o in osr_list]
ref = snr_meas[0] + 30 * np.log10(osr_list / osr_list[0])   # 1st order: 9 dB/octave
plt.figure(figsize=(7, 4))
plt.semilogx(osr_list, snr_meas, "o-", base=2, label="measured")
plt.semilogx(osr_list, ref, "--", base=2, label="9 dB / octave (1st order)")
plt.xlabel("oversampling ratio (OSR)"); plt.ylabel("in-band SNR [dB]")
plt.legend(); plt.title("1st-order shaping: ~9 dB per OSR octave")
plt.tight_layout(); plt.savefig("figures/fig4_snr_vs_osr.png", dpi=130); plt.show()

print("\nfigures written to ./figures/")
